<a href="https://colab.research.google.com/github/Monia1801/Prompt-Engineering/blob/main/Exp_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain-google-genai langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.0 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.1
)

In [4]:
llm_with_maps   = llm.bind_tools([{"google_maps": {}}])
llm_with_search = llm.bind_tools([{"google_search": {}}])

In [6]:
maps_prompt = ChatPromptTemplate.from_template(
    "Locate the following venue and return its exact address, neighborhood, "
    "and operational hours and travel plan from vizag: {venue_query}"
)
search_prompt = ChatPromptTemplate.from_template(
    "Search for recent local news, major public events, transit disruptions, "
    "or safety warnings happening today near: {venue_query}"
)
aggregator_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an Elite Travel Concierge. Synthesize the Google Maps "
     "logistical data and the Google Search situational awareness "
     "data into a comprehensive evening briefing."),
      ("human", "Venue: {venue_query}\n\n[MAPS LOGISTICS]:\n{location_data}"
       "\n\n[SITUATIONAL NEWS & SAFETY]:\n{situational_data}")
])

In [7]:
maps_chain   = maps_prompt | llm_with_maps | StrOutputParser()
search_chain = search_prompt | llm_with_search | StrOutputParser()

parallel_grounding_engine = RunnableParallel(
    venue_query      = RunnablePassthrough(),
    location_data    = maps_chain,
  situational_data = search_chain
  )
travel_briefing_pipeline = (
    parallel_grounding_engine | aggregator_prompt | llm | StrOutputParser()
)

In [8]:
query = input("Enter the Topic: ")
print("--- Executing Parallel Grounding Pipeline (Maps + Search) ---\n")
briefing = travel_briefing_pipeline.invoke(query)
print(briefing)

Enter the Topic: Narendra Modi Cricket Stadium, Ahmedabad on May 31st, 2026
--- Executing Parallel Grounding Pipeline (Maps + Search) ---

## Evening Briefing: Ahmedabad - May 31st, 2026

**Subject:** Narendra Modi Cricket Stadium Event & Travel Considerations

**Date:** May 31st, 2026

**Prepared For:** [Client Name/Designation]

**Overview:**

Tonight, the Narendra Modi Cricket Stadium in Ahmedabad is the epicenter of the highly anticipated **IPL 2026 Final**, featuring Royal Challengers Bengaluru against Gujarat Titans. While the stadium itself is a well-established venue, the surrounding environment presents several logistical and situational factors that require careful consideration for any travel plans.

**Logistical Considerations:**

*   **Venue Location:** The Narendra Modi Cricket Stadium is situated at Jawaharlal Nehru Marg, Memnagar, Ahmedabad, Gujarat 380009. It is located within the Memnagar neighborhood.
*   **Stadium Operational Hours:** As a dedicated sporting venue, 